In [1]:
%load_ext autoreload
%autoreload 2
import os
os.chdir('/workspace/RL4Auditing')
import numpy as np
import matplotlib.pyplot as plt

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import numpy as np

# Custom CNN Model
class CNN5Layer(nn.Module):
    def __init__(self, num_classes=15):
        super(CNN5Layer, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)  # Layer 1
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)  # Layer 2
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)  # Layer 3
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)  # Layer 4
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1)  # Layer 5
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)  # Downsampling
        self.relu = nn.ReLU()  # Activation function
        
        self.fc = nn.Linear(512 * 7 * 7, num_classes)  # Fully Connected Layer
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = self.pool(self.relu(self.conv4(x)))
        x = self.pool(self.relu(self.conv5(x)))
        
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)  # No activation (BCEWithLogitsLoss expects raw logits)
        return x


# Custom Dataset Class
class NPZImageDataset(torch.utils.data.Dataset):
    def __init__(self, npz_path, transform=None):
        data = np.load(npz_path)
        self.images = data['images']
        self.labels = data['labels']  # Multi-labels as a numpy array
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)  # Use float for BCE loss
        
        if len(image.shape) == 2:  # Grayscale to 3 channels
            image = image.unsqueeze(0).repeat(3, 1, 1)
        else:
            image = image.permute(2, 0, 1)  # (H, W, C) -> (C, H, W)

        if self.transform:
            image = self.transform(image)
        return image, label


# Training Code
def train_cnn(train_loader, test_loader, num_classes=15, num_epochs=10, learning_rate=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = CNN5Layer(num_classes=num_classes).to(device)
    criterion = nn.BCEWithLogitsLoss()  # For multi-label classification
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    print("Starting Training...")
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {train_loss/len(train_loader):.4f}")

        # Evaluation
        model.eval()
        total_loss = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                total_loss += loss.item()
        print(f"Test Loss: {total_loss/len(test_loader):.4f}\n")

    print("Training Completed!")
    return model


In [4]:
TRAIN_NPZ = "../vu2/RL4Auditing/split_data/train.npz"
TEST_NPZ = "../vu2/RL4Auditing/split_data/test.npz"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Data Loaders
train_dataset = NPZImageDataset(TRAIN_NPZ, transform=transform)
test_dataset = NPZImageDataset(TEST_NPZ, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Train the model
model = train_cnn(train_loader, test_loader, num_classes=15, num_epochs=10, learning_rate=0.001)

Starting Training...


KeyboardInterrupt: 

In [5]:
def evaluate_per_class_accuracy(model, test_loader, num_classes=15, threshold=0.5):
    """
    Evaluate per-class accuracy for a multi-label classification model.
    
    Parameters:
    - model (nn.Module): Trained PyTorch model.
    - test_loader (DataLoader): DataLoader for the test dataset.
    - num_classes (int): Number of classes.
    - threshold (float): Threshold to convert logits to binary predictions.

    Returns:
    - per_class_accuracy (list): List of accuracy values for each class.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()  # Set model to evaluation mode
    model.to(device)
    
    # Initialize counters
    correct_per_class = torch.zeros(num_classes).to(device)
    total_per_class = torch.zeros(num_classes).to(device)
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(images)
            predictions = (torch.sigmoid(outputs) > threshold).float()  # Apply threshold
            
            # Update counters
            correct_per_class += (predictions == labels).sum(dim=0)
            total_per_class += labels.size(0)
    
    # Calculate accuracy per class
    per_class_accuracy = (correct_per_class / total_per_class).cpu().numpy()
    
    # Print results
    for i, acc in enumerate(per_class_accuracy):
        print(f"Class {i}: Accuracy = {acc:.2f}")
    
    return per_class_accuracy

In [7]:
# Load the trained model
model = CNN5Layer(num_classes=15)
model.load_state_dict(torch.load("../vu2/RL4Auditing/split_data/cnn5layer_multi_label.pth"))

# Evaluate per-class accuracy
per_class_accuracy = evaluate_per_class_accuracy(model, test_loader, num_classes=15)

KeyboardInterrupt: 